# 01 — Ingest, QC and schema discovery

**Frangieh et al. 2021 Perturb-CITE-seq** — ~218k patient-derived melanoma
cells, 248 CRISPR-KO targets, three environments (control / IFN-γ / TIL
co-culture), RNA + 20-plex ADT.

**Goal of this notebook:** find out what is actually in these files, correct
`config.yaml` to match, and produce the power table that determines what the
rest of the project is allowed to claim.

Nothing downstream should run until `check_schema()` passes.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# quiet scanpy
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

from adjustText import adjust_text

cfg = load_config()
panels = load_panels()
P = paths(cfg)
SEED = set_seed(cfg)
apply_style(cfg)

sc.settings.verbosity = 1
print(f"repo: {P.root}")
print(f"seed: {SEED}")


## 1. Fetch

Idempotent — skips files already on disk (~3-5 GB).

In [ ]:
from src.data_io import fetch_raw
files = fetch_raw(cfg)
files

## 2. Schema discovery

The `schema:` block in `config/config.yaml` is an assumption about scPerturb's harmonised column
names. Look at it, then fix config to work with it.

In [ ]:
from src.data_io import load_rna, load_protein, describe_schema

rna = load_rna(cfg)
rna_schema = describe_schema(rna, "RNA")

In [ ]:
adt = load_protein(cfg)
adt_schema = describe_schema(adt, "ADT")
print("\nADT panel:", list(adt.var_names))

### Reconcile

Edit `config/config.yaml` → `schema:` MHC is HLA_A, etc.

In [ ]:
from src.data_io import check_schema
check_schema(rna, cfg)

### Write the observed ADT panel

`panels.yaml` has `adt_to_rna` dict matching ADT target names to gene names, and other defined sets etc.
This is where the schema reveals what strings are correct (i.e. MHC-I is HLA-A etc in this dataset).

In [ ]:
import yaml
observed = {
    "adt": {
        "observed_features": list(map(str, adt.var_names)),
        "adt_to_rna": {f: [] for f in map(str, adt.var_names)},
    }
}
out = P.root / "config" / "panels_observed.yaml"
out.write_text(yaml.safe_dump(observed, sort_keys=False))
print(f"wrote {out}\n-> fill in adt_to_rna, merge into config/panels.yaml, commit")

In [ ]:
# check the ADT signal in the metadata
adt.var[['Target', 'Clone', 'TotalSeq A#', 'Isotype_control']]

## 3. Align modalities

The two h5ads are distributed separately and are not guaranteed to contain the
same cells in the same order.

In [ ]:
from src.data_io import check_schema
check_schema(rna, cfg)

## 4. The power table

Cells per perturbation × condition. **This single figure sets the ceiling on
what the project can claim.** An arm with 12 cells will not support a
context-dependence call regardless of what any p-value says.

Note the filter is applied *per condition*, not globally: a perturbation can
be well powered in control and underpowered in co-culture (because those cells
were killed), and that asymmetry is itself informative.

In [ ]:
from src.pseudobulk import group_sizes

sizes = group_sizes(rna, cfg)
print(f"{sizes.shape[0]} perturbations x {sizes.shape[1]} conditions")
sizes.describe()

In [ ]:
print(rna.obs['MOI'].value_counts().sort_index())
print(pd.crosstab(rna.obs['MOI'], rna.obs['nperts']))
rna.obs['guide_id'].astype(str).str.contains(',').mean()

In [ ]:
# before plotting, filter dataset for single-guide receiving cells

# Restrict to unambiguous single-guide cells. Justified by panels B and C:
# 31% of cells carry >=2 guides, and no combination reaches n=30, so the
# multiplexed fraction cannot support contrasts of its own.
rna_moi1 = rna[rna.obs["MOI"] == 1].copy()
adt_moi1 = adt[rna_moi1.obs_names].copy()

sizes = group_sizes(rna_moi1, cfg)
sizes_sorted = sizes.reindex(sizes.min(axis=1).sort_values().index)

In [ ]:
guides_per_gene = rna.obs.groupby('perturbation', observed=True)['sgRNA'].nunique()
print(guides_per_gene.value_counts())

In [ ]:
# preliminary QC plots

# reload config to get colors for plotting - loaded at top of nb, but if they have changed etc
cfg = load_config()

import seaborn as sns
from adjustText import adjust_text
from src.pseudobulk import group_sizes

# ---- pull settings out of the config dict (cfg = parsed config.yaml) --------
min_n = cfg["qc"]["min_cells_per_perturbation_per_condition"]   # currently 30
pal   = condition_palette(cfg)          # {"Control": "#8A8A8A", "IFNγ": ..., ...}
cond_order = ["Control", "IFNγ", "Co-culture"]   # baseline → cytokine → killing
ctrl_label = "control"                  # <- set from guides_per_gene[guides_per_gene == 75]

# ---- THE FILTER ------------------------------------------------------------
# Restrict to cells carrying exactly one guide. Justified by panels A and B:
# 31% of cells are multiplexed, and no guide combination reaches n=30 (max 13),
# so the multiplexed fraction cannot support contrasts of its own. MOI==1 is
# also what the experiment was designed to produce.
#
# Panels A and B stay on the UNFILTERED object — they exist to justify this
# filter, so they must describe the population before it was applied.
# Panels C, D, E use rna_moi1.
rna_moi1 = rna[rna.obs["MOI"] == 1].copy()
print(f"MOI==1: {rna_moi1.n_obs:,} of {rna.n_obs:,} cells "
      f"({rna_moi1.n_obs / rna.n_obs:.0%})")

# group_sizes is from pseudobulk.py - adata.obs.groupby("perturbation","condition") unstacked into table
# groupby(perturbation1, perturbation2) %>% size() long format %>%
# unstack() pivot perturbation2 to cols %>% sort_index() alphabetizes rows (perturbation1 CRISPR target genes)
sizes = group_sizes(rna_moi1, cfg)

# re sort and index the cell counts table, sorted by min (cross all cols)
sizes_sorted = sizes.reindex(sizes.min(axis=1).sort_values().index)

# ---- mosaic format for plot: repeat a letter to widen that panel ------------
# Row 1: A (guide multiplicity), B (combination feasibility) — the filtering
#        argument, on ALL cells.
# Row 2: C (power table), D (underpowered perturbations) — post-filter.
# Row 3: E (guide balance within target) — post-filter.
fig, ax = plt.subplot_mosaic(
    """
    AABB
    CCDD
    EEEE
    """,
    figsize=(15, 17),
)

# =============================== A ==========================================
# Guides per cell, ALL cells. Bar not histogram: MOI is already an exact
# integer count, so binning would merge values that mean different things.
# Note this counts GUIDES, not targets — two guides against the same gene
# would register as MOI=2, though that is rare in a pooled library this size.
moi = rna.obs["MOI"].value_counts().sort_index()

# colour by MOI: interested in MOI=1.  all with >1 have to be discarded.
colors = ["#bbbbbb" if m == 0 else "#4c9f70" if m == 1 else "#535353"
          for m in moi.index]

# moi is the series from the above value_counts()
ax["A"].bar(moi.index, moi.values, color=colors, width=0.8)
ax["A"].set_yscale("log")               # 127k vs 2 — linear would erase the tail
ax["A"].set_xlabel("number of guides detected per cell (MOI)")
ax["A"].set_ylabel("number of cells (log)")
ax["A"].set_xticks(range(0, moi.index.max() + 1, 2))

n0    = moi.get(0, 0)
n1    = moi.get(1, 0)
nmult = moi[moi.index >= 2].sum()
tot   = moi.sum()
ax["A"].text(0.97, 0.95,
             f"MOI 0:  {n0:,} ({n0/tot:.0%})\n"
             f"MOI 1:  {n1:,} ({n1/tot:.0%})\n"
             f"MOI ≥2: {nmult:,} ({nmult/tot:.0%})",
             transform=ax["A"].transAxes, ha="right", va="top",
             fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="none", alpha=0.85))

# =============================== B ==========================================
# Follows from A: 31% of cells carry >=2 guides. Could those cells support
# gene x gene contrasts? Each bar is one observed guide COMBINATION, and its
# length is how many cells carry it. The best-represented combination in the
# entire dataset has 13 cells — n=30 is never approached. Multiplexing here is
# accidental co-infection scattered across ~30,600 possible pairs, not a
# designed combinatorial library. Nothing to rescue; MOI>=2 cells are dropped.
multi = rna.obs[rna.obs["MOI"] >= 2]
pair_counts = multi["guide_id"].value_counts()

top = pair_counts.head(30)              # 30 best-represented combinations
y = np.arange(len(top))

ax["B"].barh(y, top.values, color="#535353", height=0.8)
ax["B"].axvline(min_n, ls="--", c="grey", lw=1.2, label=f"n = {min_n}")
ax["B"].set_yticks([])                  # combination IDs are meaningless strings
ax["B"].invert_yaxis()                  # best at top
ax["B"].set_xlabel("cells sharing this guide combination")
ax["B"].set_ylabel("unique guide combination (top 30)")
ax["B"].set_xlim(0, min_n * 1.15)       # scale to the threshold, not the data
ax["B"].legend(fontsize=8, loc="upper right")

ax["B"].text(0.97, 0.05,
             f"unique guide combinations: {len(pair_counts):,}\n"
             f"max n: {pair_counts.max()}",
             transform=ax["B"].transAxes, ha="right", va="bottom",
             fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="none", alpha=0.85))

# =============================== C ==========================================
# POST-FILTER power table. Sorted rank plot: every perturbation ordered by its
# WORST condition. Reading position beats reading colour, so this is preferable
# to a heatmap for spotting the low tail.
#
# sizes:
# rna_moi1.obs %>% groupby(perturbation, perturbation_2) — lazy, nothing computed
# %>% size() cells per bucket → Series w/ MultiIndex, long format
# %>% unstack(fill_value=0) conditions to columns → wide, 249 x 3
#     fill_value converts "zero cells observed" from NaN to 0
for cond in cond_order:
    ax["C"].plot(range(len(sizes_sorted)), sizes_sorted[cond],
                 lw=1.2, alpha=0.85, label=cond, color=pal.get(cond, "#888"))

ax["C"].axhline(min_n, ls="--", c="grey", lw=1.2, label=f"n = {min_n} cells")
ax["C"].set_yscale("log")               # spans ~1 to ~1000; log makes tail legible
ax["C"].set_xlabel("perturbation (sorted by worst-condition n)")
ax["C"].set_ylabel("number of cells in bin (log)")
ax["C"].legend(fontsize=8, loc="lower right")

# Label the five worst perturbations — these are the calls you'd have to defend
texts = []
for i, gene in enumerate(sizes_sorted.index[:5]):
    ymin = sizes_sorted.loc[gene].min()
    texts.append(ax["C"].text(i, ymin, f"{gene} (n={int(ymin)})", fontsize=7))

adjust_text(texts, ax=ax["C"],
            arrowprops=dict(arrowstyle="-", lw=0.5, color="grey"),
            expand_points=(1.5, 1.5))

# =============================== D ==========================================
# POST-FILTER. Perturbations failing the cell-count threshold in AT LEAST ONE
# condition. .any(axis=1) → True if any of the 3 columns is below min_n.
under = sizes[(sizes < min_n).any(axis=1)]
under = under.reindex(under.min(axis=1).sort_values().index)   # worst first

x = np.arange(len(under))
w = 0.26                                 # bar width; 3 bars must fit in 1 unit
for j, cond in enumerate(cond_order):
    ax["D"].bar(x + (j - 1) * w, under[cond], width=w,
                label=cond, color=pal.get(cond, "#888"))

ax["D"].axhline(min_n, ls="--", c="grey", lw=1.2)
ax["D"].set_xticks(x)
ax["D"].set_xticklabels(under.index, rotation=90, fontsize=7)
ax["D"].set_ylabel("number of cells")
ax["D"].legend(fontsize=8)

# =============================== E ==========================================
# POST-FILTER. sgRNA representation within each target. Each bar is one gene;
# stacked segments are its guides as a proportion of that gene's cells.
# Perfect balance = four even quarters.
#
# Why this matters: a hit driven by one guide is an off-target effect until
# independent guides agree. A gene where one guide holds most of the cells
# cannot support that cross-check, regardless of total cell count.
#
# Control excluded — it pools ~75 non-targeting guides under one label, so its
# "balance" is not the quantity being measured here.
gc = (rna_moi1.obs[rna_moi1.obs["perturbation"] != ctrl_label]
      .groupby(["perturbation", "sgRNA"], observed=True)
      .size().unstack(fill_value=0))

prop = gc.div(gc.sum(axis=1), axis=0)                  # rows sum to 1

# Sort each gene's guides by size so segment order is descending everywhere.
# Guide identity is meaningless across genes; the shape is what's readable.
sorted_props = np.sort(prop.values, axis=1)[:, ::-1]   # descending per row

# unstack() creates one column per sgRNA across ALL genes (~800), so nearly
# every column is zero for any given gene. Keep only as many as the widest
# gene actually uses.
max_guides = int((sorted_props > 0).sum(axis=1).max())
sorted_props = sorted_props[:, :max_guides]
print(f"max guides per gene (MOI==1): {max_guides}")

order = np.argsort(-sorted_props[:, 0])                # most-dominated first
sorted_props_all = sorted_props[order]
gene_order_all = prop.index[order]

# Show the 100 most-dominated genes; the balanced remainder is summarised in
# the text box rather than plotted.
N_SHOW = 100
sorted_props = sorted_props_all[:N_SHOW]
gene_order = gene_order_all[:N_SHOW]

x = np.arange(len(gene_order))
bottom = np.zeros(len(gene_order))
blues = ["#0B3D6B", "#1F6FB2", "#5BA3D9", "#B3D4EC"]   # dark → light
rank_labels = ["1st", "2nd", "3rd", "4th"]

for k in range(sorted_props.shape[1]):
    lbl = f"{rank_labels[k]} most-represented guide" if k < len(rank_labels) else None
    ax["E"].bar(x, sorted_props[:, k], bottom=bottom, width=0.78,
                color=blues[k % len(blues)], linewidth=0, label=lbl)
    bottom += sorted_props[:, k]

ax["E"].axhline(1/3, ls="--", c="crimson", lw=1.2, label="even split (3 guides)")
ax["E"].set_xlim(-0.7, len(gene_order) - 0.3)
ax["E"].set_ylim(0, 1)
ax["E"].set_xticks(x)
ax["E"].set_xticklabels(gene_order, rotation=90, fontsize=5)
ax["E"].set_xlabel(f"target gene — {N_SHOW} most guide-dominated of {len(gene_order_all)}")
ax["E"].set_ylabel("proportion of gene's cells")
ax["E"].legend(fontsize=7, loc="lower right", ncol=2, framealpha=0.9)

top_all = sorted_props_all[:, 0]
ax["E"].text(0.005, 0.05,
             f"genes:            {len(gene_order_all)}\n"
             f"top guide >50%:   {(top_all > 0.5).sum()}\n"
             f"top guide >75%:   {(top_all > 0.75).sum()}\n"
             f"median top guide: {np.median(top_all):.2f}",
             transform=ax["E"].transAxes, ha="left", va="bottom",
             fontsize=8, family="monospace", color="white",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#1a1a1a",
                       edgecolor="none", alpha=0.85))

# ---- panel letters + titles, both left-aligned in one call ------------------
titles = {
    "A": "CRISPR guides per cell (all cells)",
    "B": "no guide combination reaches usable depth",
    "C": "cells per perturbation (pre-filt for MOI=1)",
    "D": f"{len(under)} perturbations below n={min_n} in ≥1 condition (MOI=1)",
    "E": "sgRNA representation within target (MOI=1)",
}
for label, a in ax.items():
    a.set_title(f"{label}) {titles.get(label, '')}", loc="left",
                fontweight="bold", fontsize=12, pad=8)

fig.tight_layout()
savefig(fig, "01_qc_overview", cfg)      # → results/figures/01_qc_overview.png

In [ ]:
multi = rna.obs[rna.obs['MOI'] >= 2]
pair_counts = multi['guide_id'].value_counts()   # or however combos are encoded
print(pair_counts.describe())
print((pair_counts >= 30).sum())

## 5. Cell-level QC

scPerturb already applied a uniform QC pass, so thresholds here are
deliberately permissive. Tighten only with a plot that justifies it.

Expect the co-culture arm to look different — those cells are under attack - likely elevated pct.mt etc

In [ ]:
s = cfg["schema"]["obs"]
qc_cols = [s["n_counts"], s["n_genes"], s["percent_mito"]]
qc_cols = [c for c in qc_cols if c in rna.obs.columns]

fig, axes = plt.subplots(1, len(qc_cols), figsize=(4.5 * len(qc_cols), 4))
for ax, col in zip(np.atleast_1d(axes), qc_cols):
    sns.violinplot(data=rna.obs, x=s["condition"], y=col, ax=ax,
                   palette=condition_palette(cfg), cut=0)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
savefig(fig, "01_qc_by_condition", cfg)

### Guide multiplicity

Expect a low-MOI design (one guide per cell). Cells with more than one assigned guide confound every downstream contrast.

In [ ]:
# =============================================================================
# Cell-level QC, before vs after the MOI==1 filter
#
# scPerturb applied uniform QC across all 44 datasets in the harmonised
# release, so the "before" column is already post-QC by their thresholds.
# We trust that and add no further cell-level cutoffs.
#
# The filter being evaluated here is therefore MOI==1, not a quality cutoff.
# The question this figure answers: do the 42% of cells we are discarding
# differ systematically in quality from the ones we keep? If the right column
# is a clean subset of the left, the filter is unbiased with respect to cell
# quality and we can attribute any downstream difference to the design issue
# rather than to a quality confound.
#
# NOTE ON percent_mito: elevated mito fraction in the co-culture arm is
# EXPECTED — those cells are being killed by TILs. Do not filter it away.
# A mito threshold tuned to the control arm would preferentially delete the
# cells that responded to T-cell attack, biasing every co-culture contrast
# toward survivors.
# =============================================================================

qc_metrics = [
    ("ncounts",      "UMIs per cell",        True),   # (column, label, log-y)
    ("ngenes",       "genes per cell",       True),
    ("percent_mito", "% mitochondrial",      False),
    ("percent_ribo", "% ribosomal",          False),
]
# keep only metrics that actually exist in .obs
qc_metrics = [(c, l, lg) for c, l, lg in qc_metrics if c in rna.obs.columns]

s_cond = cfg["schema"]["obs"]["condition"]     # "perturbation_2"

fig, axes = plt.subplots(len(qc_metrics), 2,
                         figsize=(12, 3.1 * len(qc_metrics)),
                         sharey="row")

panels = [
    (f"all cells (n = {rna.n_obs:,})",     rna.obs),
    (f"MOI = 1 (n = {rna_moi1.n_obs:,})",  rna_moi1.obs),
]

for i, (col, label, logy) in enumerate(qc_metrics):
    for j, (title, obs) in enumerate(panels):
        a = axes[i, j]

        sns.violinplot(data=obs, x=s_cond, y=col, order=cond_order,
                       hue=s_cond, hue_order=cond_order, palette=pal,
                       legend=False, cut=0, linewidth=0.8, ax=a)

        # median as a white tick — violins hide it at this width
        med = obs.groupby(s_cond, observed=True)[col].median().reindex(cond_order)
        a.scatter(range(len(cond_order)), med.values,
                  color="white", s=14, zorder=10, edgecolor="black", linewidth=0.5)

        if logy:
            a.set_yscale("log")
        a.set_ylabel(label if j == 0 else "")
        a.set_xlabel("")
        a.tick_params(axis="x", rotation=20)

        # n per group along the bottom
        n = obs[s_cond].value_counts().reindex(cond_order)
        a.set_xticks(range(len(cond_order)))
        a.set_xticklabels([f"{c}\nn={n[c]:,}" for c in cond_order], fontsize=8)

        if i == 0:
            a.set_title(f"{'AB'[j]}) {title}", loc="left",
                        fontweight="bold", fontsize=12, pad=8)

fig.suptitle("Cell-level QC: inherited scPerturb filtering, before vs after MOI=1",
             fontsize=13, fontweight="bold", y=0.995)
fig.tight_layout()
savefig(fig, "01_qc_violins_moi_filter", cfg)


# ---- numeric comparison: does the filter shift any metric? ------------------
# Eyeballing violins is unreliable for subtle shifts. Print the medians.
rows = []
for col, label, _ in qc_metrics:
    for cond in cond_order:
        before = rna.obs.loc[rna.obs[s_cond] == cond, col]
        after  = rna_moi1.obs.loc[rna_moi1.obs[s_cond] == cond, col]
        rows.append({
            "metric": col,
            "condition": cond,
            "median_all": before.median(),
            "median_moi1": after.median(),
            "pct_change": 100 * (after.median() - before.median()) / before.median(),
        })
qc_shift = pd.DataFrame(rows)
print(qc_shift.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

## 6. Save filtered dataset

In [ ]:
# =============================== 6. Write =====================================
# Apply filters and persist the paired object. Everything downstream
# (nb02-05) loads this file rather than the raw h5ads, so the MOI filter and
# modality alignment happen exactly once.

from src.data_io import align_modalities

# Gene filter: drop genes detected in too few cells. Cell-level QC is
# inherited from scPerturb; no additional cutoffs applied (see violins above).
sc.pp.filter_genes(rna_moi1, min_cells=cfg["qc"]["min_cells_per_gene"])
print(f"genes retained: {rna_moi1.n_vars:,}")

# Match the ADT object to the surviving cells
rna_f, adt_f = align_modalities(rna_moi1, adt)

# Preserve the authors' UMAP coordinates as a reference embedding
if {"umap_x", "umap_y"}.issubset(rna_f.obs.columns):
    rna_f.obsm["X_umap_orig"] = rna_f.obs[["umap_x", "umap_y"]].to_numpy()

# Keep raw counts safe before any normalisation downstream
rna_f.layers["counts"] = rna_f.X.copy()
adt_f.layers["counts"] = adt_f.X.copy()

import mudata as md
mdata = md.MuData({"rna": rna_f, "adt": adt_f})
out = P.data_interim / "frangieh_qc.h5mu"
mdata.write(out)
print(f"\nwrote {out}")
print(mdata)

In [ ]:
import session_info
session_info.show()